In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

class Paths:
    p = "/Users/shirokoshikentaro/Desktop/Python3/house-prices-advanced-regression-techniques"
    train = p + "/train.csv"
    test = p + "/test.csv"
    sample = p + "/sample_submission.csv"

# ===== ステップ1: データ読み込み =====
train = pd.read_csv(Paths.train)
test = pd.read_csv(Paths.test)

print(f"train shape: {train.shape}")
print(f"test shape: {test.shape}")

# ===== ステップ2: 目的変数を先に取り出す =====
y_train = np.log1p(train["SalePrice"].values)
print(f"y_train shape: {y_train.shape}")

# Id列を保存
train_id = train["Id"]
test_id = test["Id"]

# ステップ2の直後に追加
# GrLivAreaが大きいのに価格が低い物件を除外
train = train[~((train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000))]
y_train = np.log1p(train["SalePrice"].values)
train_id = train["Id"]

# ===== ステップ3: 新規特徴量作成 =====
for df in [train, test]:
    # 既存の特徴量
    df["QualGrLiv"] = df["OverallQual"] * df["GrLivArea"]
    df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
    df["QualTotalSF"] = df["OverallQual"] * df["TotalSF"]
    df["Age"] = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
    df["TotalBath"] = (df["FullBath"] + 0.5*df["HalfBath"].fillna(0) + 
                       df["BsmtFullBath"].fillna(0) + 0.5*df["BsmtHalfBath"].fillna(0))
    df["AreaPerRoom"] = df["GrLivArea"] / df["TotRmsAbvGrd"].replace(0, 1)
    df["GarageScore"] = df["GarageCars"] * df["GarageArea"]

    # log変換した特徴量
    df["Log_GrLivArea"] = np.log1p(df["GrLivArea"])
    df["Log_LotArea"] = np.log1p(df["LotArea"])
    df["Log_TotalSF"] = np.log1p(df["TotalSF"])
    
    # 新規特徴量
    df["QualGrLivLog"] = df["OverallQual"] * df["Log_GrLivArea"]
    df["TotalPorchSF"] = (df["OpenPorchSF"] + df["EnclosedPorch"] + 
                          df["3SsnPorch"].fillna(0) + df["ScreenPorch"].fillna(0))
    df["IsNew"] = (df["Age"] <= 5).astype(int)
    df["IsRemodeled"] = (df["YearRemodAdd"] != df["YearBuilt"]).astype(int)
    df["HasBsmt"] = (df["TotalBsmtSF"] > 0).astype(int)
    df["BsmtScore"] = df["TotalBsmtSF"] * df["HasBsmt"]

# ===== ステップ4: Id, SalePriceを削除 =====
train = train.drop(["Id", "SalePrice"], axis=1)
test = test.drop(["Id"], axis=1)

# ===== ステップ5: 数値とカテゴリを自動判定 =====
# 数値型のカラムを取得
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
# カテゴリカルのカラムを取得（object型）
cat_cols = train.select_dtypes(include=['object']).columns.tolist()

print(f"数値特徴量: {len(num_cols)}個")
print(f"カテゴリ特徴量: {len(cat_cols)}個")
print(f"カテゴリカル変数: {cat_cols}")

# ===== ステップ6: 欠損値処理 =====
print("\n欠損値処理中...")
for col in num_cols:
    train[col] = train[col].fillna(0)
    test[col] = test[col].fillna(0)

for col in cat_cols:
    train[col] = train[col].fillna("None")
    test[col] = test[col].fillna("None")

# ===== ステップ7: One-Hotエンコーディング =====
print("One-Hotエンコーディング中...")

# trainとtestを結合
train['is_train'] = 1
test['is_train'] = 0
combined = pd.concat([train, test], axis=0, ignore_index=True)

# One-Hotエンコーディング（全カテゴリカル変数を対象）
combined_encoded = pd.get_dummies(
    combined, 
    columns=cat_cols,
    drop_first=True,
    dtype=int
)

# trainとtestに分割
train_encoded = combined_encoded[combined_encoded['is_train'] == 1].drop('is_train', axis=1)
test_encoded = combined_encoded[combined_encoded['is_train'] == 0].drop('is_train', axis=1)

# 特徴量リスト
all_features = list(train_encoded.columns)
print(f"One-Hot後の総特徴量: {len(all_features)}個")

# ===== ステップ8: X_train, X_test作成 =====
X_train = train_encoded.values
X_test = test_encoded.values

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")

# ===== ステップ9: CV =====
params = {
    "boosting_type": "gbdt",
    "objective": "regression",
    "metric": "rmse",
    "num_leaves": 16,  # 31→16で過学習抑制
    "learning_rate": 0.03,  # 0.05→0.03でより慎重に
    "n_estimators": 15000,
    "min_child_samples": 30,  # 20→30
    "subsample": 0.7,  # 0.8→0.7
    "colsample_bytree": 0.7,  # 0.8→0.7
    "reg_alpha": 0.5,  # 0.1→0.5で正則化強化
    "reg_lambda": 0.5,
    "random_state": 123,
    "verbose": -1,
}

cv = KFold(n_splits=5, shuffle=True, random_state=123)
metrics = []

print("\n===== Cross Validation =====")
for fold, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
    X_val, y_val = X_train[val_idx], y_train[val_idx]
    
    model = lgb.LGBMRegressor(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    
    y_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    metrics.append(rmse)
    print(f"Fold {fold}: {rmse:.5f}")

print(f"\n[CV] {np.mean(metrics):.5f}±{np.std(metrics):.5f}")

# ===== ステップ10: 全データで学習 & 予測 =====
model = lgb.LGBMRegressor(**params)

model.fit(X_train, y_train, 
          eval_set=[(X_train, y_train)],
          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)])

y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)

# ===== ステップ11: 提出ファイル =====
submission = pd.DataFrame({
    "Id": test_id,
    "SalePrice": y_pred
})

submission.to_csv("submission_onehot.csv", index=False)
print(f"\n✅ 保存完了！平均予測価格: ${y_pred.mean():,.0f}")

train shape: (1460, 81)
test shape: (1459, 80)
y_train shape: (1460,)
数値特徴量: 53個
カテゴリ特徴量: 43個
カテゴリカル変数: ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']

欠損値処理中...
One-Hotエンコーディング中...
One-Hot後の総特徴量: 283個

X_train shape: (1458, 283)
X_test shape: (1459, 283)
y_train shape: (1458,)

===== Cross Validation =====
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[427]	valid_0's rmse: 0.111947
Fold 0: 0.11195
Training until